In [ ]:
import os
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

# --- User config (edit these) ---
IMAGES_DIR = r"D:\First try2\justtry\images\train"          # TODO: set your image folder
ANNOTATIONS_JSON = r"E:\code\training code\instances_Train.json"  # TODO: set your COCO json
OUTPUT_DIR = r"E:\code\training code\runs-fasterrcnn"

# Training
EPOCHS = 60
BATCH = 2               # keep small for 8GB VRAM
ACCUMULATE = 1          # set >1 for gradient accumulation
LR = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
STEP_SIZE = 8
GAMMA = 0.1
NUM_WORKERS = 0
AMP = True              # mixed precision to reduce VRAM

# Model input sizes (defaults are 800/1333); adjust only if needed
MIN_SIZE = 800
MAX_SIZE = 1333

# --- End user config ---


class CocoDetectionMapped(Dataset):
    def __init__(self, images_dir, ann_file, transforms=None):
        from pycocotools.coco import COCO

        self.images_dir = images_dir
        self.coco = COCO(ann_file)
        self.img_ids = sorted(self.coco.getImgIds())
        self.img_ids = [img_id for img_id in self.img_ids
                if len(self.coco.getAnnIds(imgIds=img_id)) > 0]
        # Build category id -> contiguous id mapping
        cat_ids = sorted(self.coco.getCatIds())
        self.cat_id_to_contig = {cat_id: i + 1 for i, cat_id in enumerate(cat_ids)}
        self.contig_to_cat_id = {v: k for k, v in self.cat_id_to_contig.items()}
        self.num_classes = len(cat_ids) + 1  # +1 for background

        self.transforms = transforms

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.images_dir, img_info["file_name"])

        # Load image
        from PIL import Image
        image = Image.open(img_path).convert("RGB")

        # Load annotations
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 1 or h <= 1:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(self.cat_id_to_contig[ann["category_id"]])
            areas.append(ann.get("area", w * h))
            iscrowd.append(ann.get("iscrowd", 0))

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([img_id]),
            "area": areas,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            image = self.transforms(image)

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


def train_one_epoch(model, optimizer, data_loader, device, scaler, accumulate):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    print("Entered train_one_epoch. num_batches =", len(data_loader))
    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with torch.cuda.amp.autocast(enabled=scaler is not None):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            loss = losses / accumulate

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (i+1) % accumulate == 0:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    transforms = T.Compose([T.ToTensor()])
    dataset = CocoDetectionMapped(IMAGES_DIR, ANNOTATIONS_JSON, transforms=transforms)
    data_loader = DataLoader(
        dataset,
        batch_size=BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=collate_fn,
    )

    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights, min_size=MIN_SIZE, max_size=MAX_SIZE)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    #model.roi_heads.box_predictor = torch.nn.Linear(in_features, dataset.num_classes)
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, dataset.num_classes)

    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

    scaler = torch.cuda.amp.GradScaler() if AMP and device.type == "cuda" else None

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for epoch in range(EPOCHS):
        train_one_epoch(model, optimizer, data_loader, device, scaler, ACCUMULATE)
        lr_scheduler.step()

        # Save checkpoint each epoch
        ckpt_path = os.path.join(OUTPUT_DIR, f"fasterrcnn_epoch_{epoch + 1}.pth")
        torch.save(model.state_dict(), ckpt_path)

    print("Training complete.")


if __name__ == "__main__":
    main()


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


C:\Users\asus-\AppData\Local\Temp\ipykernel_18872\723742949.py:163: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if AMP and device.type == "cuda" else None


Entered train_one_epoch. num_batches = 39


C:\Users\asus-\AppData\Local\Temp\ipykernel_18872\723742949.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batches = 39
Entered train_one_epoch. num_batch

In [8]:
import os
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

# --- User config (edit these) ---
IMAGES_DIR = r"D:\First try3\justtry\images\train"          # TODO: set your image folder
ANNOTATIONS_JSON = r"E:\code\training code\instances_Train_synthetic.json"  # TODO: set your COCO json
OUTPUT_DIR = r"E:\code\training code\runs-fasterrcnn"

# Training
EPOCHS = 60
BATCH = 2               # keep small for 8GB VRAM
ACCUMULATE = 1          # set >1 for gradient accumulation
LR = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
STEP_SIZE = 8
GAMMA = 0.1
NUM_WORKERS = 0
AMP = True              # mixed precision to reduce VRAM

# Model input sizes (defaults are 800/1333); adjust only if needed
MIN_SIZE = 800
MAX_SIZE = 1333

# --- End user config ---


class CocoDetectionMapped(Dataset):
    def __init__(self, images_dir, ann_file, transforms=None):
        from pycocotools.coco import COCO

        self.images_dir = images_dir
        self.coco = COCO(ann_file)
        self.img_ids = sorted(self.coco.getImgIds())
        self.img_ids = [img_id for img_id in self.img_ids
                if len(self.coco.getAnnIds(imgIds=img_id)) > 0]
        # Build category id -> contiguous id mapping
        cat_ids = sorted(self.coco.getCatIds())
        self.cat_id_to_contig = {cat_id: i + 1 for i, cat_id in enumerate(cat_ids)}
        self.contig_to_cat_id = {v: k for k, v in self.cat_id_to_contig.items()}
        self.num_classes = len(cat_ids) + 1  # +1 for background

        self.transforms = transforms

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.images_dir, img_info["file_name"])

        # Load image
        from PIL import Image
        image = Image.open(img_path).convert("RGB")

        # Load annotations
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 1 or h <= 1:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(self.cat_id_to_contig[ann["category_id"]])
            areas.append(ann.get("area", w * h))
            iscrowd.append(ann.get("iscrowd", 0))

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([img_id]),
            "area": areas,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            image = self.transforms(image)

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


def train_one_epoch(model, optimizer, data_loader, device, scaler, accumulate):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    print("Entered train_one_epoch. num_batches =", len(data_loader))
    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with torch.cuda.amp.autocast(enabled=scaler is not None):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            loss = losses / accumulate

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (i+1) % accumulate == 0:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    transforms = T.Compose([T.ToTensor()])
    dataset = CocoDetectionMapped(IMAGES_DIR, ANNOTATIONS_JSON, transforms=transforms)
    data_loader = DataLoader(
        dataset,
        batch_size=BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=collate_fn,
    )

    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights, min_size=MIN_SIZE, max_size=MAX_SIZE)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    #model.roi_heads.box_predictor = torch.nn.Linear(in_features, dataset.num_classes)
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, dataset.num_classes)

    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

    scaler = torch.cuda.amp.GradScaler() if AMP and device.type == "cuda" else None

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for epoch in range(EPOCHS):
        train_one_epoch(model, optimizer, data_loader, device, scaler, ACCUMULATE)
        lr_scheduler.step()

        # Save checkpoint each epoch
        ckpt_path = os.path.join(OUTPUT_DIR, f"fasterrcnn_epoch_{epoch + 1}.pth")
        torch.save(model.state_dict(), ckpt_path)

    print("Training complete.")


if __name__ == "__main__":
    main()


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


C:\Users\asus-\AppData\Local\Temp\ipykernel_18872\3515185250.py:163: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if AMP and device.type == "cuda" else None


Entered train_one_epoch. num_batches = 41


C:\Users\asus-\AppData\Local\Temp\ipykernel_18872\3515185250.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batches = 41
Entered train_one_epoch. num_batch

In [1]:
import os
import random
from dataclasses import dataclass
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset

from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models import ResNet50_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator, RPNHead

from torchvision.transforms import ColorJitter, GaussianBlur
from torchvision.transforms import functional as F


# -----------------------
# Config
# -----------------------
@dataclass
class Config:
    # Paths
    images_dir: str = r"D:\First try2\justtry\images\train"
    annotations_json: str = r"E:\code\training code\instances_Train.json"
    output_dir: str = r"E:\code\training code\runs-fasterrcnn-manual"

    # Training
    epochs: int = 200
    batch_size: int = 1  # keep small for 8GB VRAM
    accumulate: int = 8
    num_workers: int = 0
    amp: bool = True

    # Optimizer
    lr: float = 0.002
    weight_decay: float = 0.0005
    momentum: float = 0.9

    # Scheduler
    warmup_epochs: int = 3
    min_lr: float = 1e-5

    # Input size (Faster R-CNN internal resize)
    min_size: int = 960
    max_size: int = 1536

    # Augment (more conservative, lighting-robust)
    hflip_p: float = 0.5
    vflip_p: float = 0.2
    rot90_p: float = 0.35

    color_jitter_p: float = 0.6
    color_jitter: tuple = (0.12, 0.12, 0.08, 0.02)  # B,C,S,H

    gamma_p: float = 0.25
    gamma_range: tuple = (0.9, 1.15)

    blur_p: float = 0.05
    blur_kernel: int = 3

    noise_p: float = 0.10
    noise_std: float = 0.01

    # Anchors (5 FPN levels)
    anchor_sizes_per_level: tuple = ((8,), (16,), (32,), (64,), (128,))
    anchor_ratios: tuple = (0.5, 1.0, 2.0)

    # Detection tuning for crowded/small objects
    rpn_pre_nms_top_n_train: int = 4000
    rpn_pre_nms_top_n_test: int = 2000
    rpn_post_nms_top_n_train: int = 2000
    rpn_post_nms_top_n_test: int = 1000
    rpn_fg_iou_thresh: float = 0.7
    rpn_bg_iou_thresh: float = 0.3
    box_score_thresh: float = 0.0
    box_nms_thresh: float = 0.5
    box_detections_per_img: int = 200

    # Backbone
    trainable_backbone_layers: int = 5  # 0-5 for resnet50_fpn_backbone

    seed: int | None = 42


# -----------------------
# Dataset: COCO -> contiguous labels
# -----------------------
class CocoDetectionMapped(Dataset):
    def __init__(self, images_dir, ann_file, transforms=None):
        from pycocotools.coco import COCO

        self.images_dir = images_dir
        self.coco = COCO(ann_file)
        self.img_ids = sorted(self.coco.getImgIds())
        # keep images that have at least one annotation
        self.img_ids = [
            img_id
            for img_id in self.img_ids
            if len(self.coco.getAnnIds(imgIds=img_id)) > 0
        ]

        cat_ids = sorted(self.coco.getCatIds())
        self.cat_id_to_contig = {cat_id: i + 1 for i, cat_id in enumerate(cat_ids)}
        self.contig_to_cat_id = {v: k for k, v in self.cat_id_to_contig.items()}
        self.num_classes = len(cat_ids) + 1  # +1 for background

        self.transforms = transforms

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.images_dir, img_info["file_name"])

        from PIL import Image

        image = Image.open(img_path).convert("RGB")

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 1 or h <= 1:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(self.cat_id_to_contig[ann["category_id"]])
            areas.append(ann.get("area", w * h))
            iscrowd.append(ann.get("iscrowd", 0))

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([img_id]),
            "area": areas,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


# -----------------------
# Transforms
# -----------------------
class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target


class ToTensor:
    def __call__(self, image, target):
        return F.to_tensor(image), target


class ColorJitterOnly:
    def __init__(self, p, jitter):
        self.p = p
        self.jitter = ColorJitter(*jitter)

    def __call__(self, image, target):
        if random.random() < self.p:
            image = self.jitter(image)
        return image, target


class RandomGamma:
    def __init__(self, p, gamma_range):
        self.p = p
        self.gamma_range = gamma_range

    def __call__(self, image, target):
        if random.random() < self.p:
            gamma = random.uniform(*self.gamma_range)
            image = F.adjust_gamma(image, gamma)
        return image, target


class RandomGaussianBlur:
    def __init__(self, p, k):
        self.p = p
        self.blur = GaussianBlur(kernel_size=k)

    def __call__(self, image, target):
        if random.random() < self.p:
            image = self.blur(image)
        return image, target


class RandomNoise:
    def __init__(self, p, std):
        self.p = p
        self.std = std

    def __call__(self, image, target):
        if random.random() < self.p:
            noise = torch.randn_like(image) * self.std
            image = torch.clamp(image + noise, 0.0, 1.0)
        return image, target


class RandomHorizontalFlip:
    def __init__(self, p):
        self.p = p

    def __call__(self, image, target):
        if random.random() < self.p:
            _, h, w = image.shape
            image = F.hflip(image)
            boxes = target["boxes"].clone()
            boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
            target["boxes"] = boxes
        return image, target


class RandomVerticalFlip:
    def __init__(self, p):
        self.p = p

    def __call__(self, image, target):
        if random.random() < self.p:
            _, h, w = image.shape
            image = F.vflip(image)
            boxes = target["boxes"].clone()
            boxes[:, [1, 3]] = h - boxes[:, [3, 1]]
            target["boxes"] = boxes
        return image, target


def _rotate_points(x, y, w, h, k):
    if k == 1:  # 90 CCW
        return y, w - x
    if k == 2:  # 180
        return w - x, h - y
    if k == 3:  # 270 CCW
        return h - y, x
    return x, y


def _rotate_boxes_90(boxes, w, h, k):
    if boxes.numel() == 0:
        return boxes
    x1, y1, x2, y2 = boxes.unbind(dim=1)
    xs = torch.stack([x1, x1, x2, x2], dim=1)
    ys = torch.stack([y1, y2, y1, y2], dim=1)
    x_rot, y_rot = _rotate_points(xs, ys, w, h, k)
    x_min = x_rot.min(dim=1).values
    x_max = x_rot.max(dim=1).values
    y_min = y_rot.min(dim=1).values
    y_max = y_rot.max(dim=1).values
    return torch.stack([x_min, y_min, x_max, y_max], dim=1)


class RandomRotate90:
    def __init__(self, p):
        self.p = p

    def __call__(self, image, target):
        if random.random() < self.p:
            k = random.choice([1, 2, 3])
            _, h, w = image.shape
            image = torch.rot90(image, k, dims=(1, 2))
            target["boxes"] = _rotate_boxes_90(target["boxes"], w, h, k)
        return image, target


def build_transforms(cfg: Config):
    # NOTE: keep ToTensor early so later ops run on tensors
    return Compose(
        [
            ToTensor(),
            ColorJitterOnly(cfg.color_jitter_p, cfg.color_jitter),
            RandomGamma(cfg.gamma_p, cfg.gamma_range),
            RandomGaussianBlur(cfg.blur_p, cfg.blur_kernel),
            RandomNoise(cfg.noise_p, cfg.noise_std),
            RandomHorizontalFlip(cfg.hflip_p),
            RandomVerticalFlip(cfg.vflip_p),
            RandomRotate90(cfg.rot90_p),
        ]
    )


# -----------------------
# Train
# -----------------------
def train_one_epoch(model, optimizer, data_loader, device, scaler, accumulate):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running = 0.0
    step = 0

    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with torch.cuda.amp.autocast(enabled=scaler is not None):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            loss = losses / accumulate

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (i + 1) % accumulate == 0:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        running += losses.item()
        step += 1

    return running / max(step, 1)


def build_model(cfg: Config, num_classes: int) -> torch.nn.Module:
    # Backbone: ResNet50 + FPN, using ImageNet pretrained weights
    backbone = resnet_fpn_backbone(
        backbone_name="resnet50",
        weights=ResNet50_Weights.DEFAULT,
        trainable_layers=cfg.trainable_backbone_layers,
    )

    # Anchor generator: 5 FPN levels
    anchor_generator = AnchorGenerator(
        sizes=cfg.anchor_sizes_per_level,
        aspect_ratios=(cfg.anchor_ratios,) * len(cfg.anchor_sizes_per_level),
    )

    # RPN head must match anchors per location
    rpn_head = RPNHead(
        backbone.out_channels,
        anchor_generator.num_anchors_per_location()[0],
        conv_depth=2,
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        rpn_head=rpn_head,
        min_size=cfg.min_size,
        max_size=cfg.max_size,
        rpn_pre_nms_top_n_train=cfg.rpn_pre_nms_top_n_train,
        rpn_pre_nms_top_n_test=cfg.rpn_pre_nms_top_n_test,
        rpn_post_nms_top_n_train=cfg.rpn_post_nms_top_n_train,
        rpn_post_nms_top_n_test=cfg.rpn_post_nms_top_n_test,
        rpn_fg_iou_thresh=cfg.rpn_fg_iou_thresh,
        rpn_bg_iou_thresh=cfg.rpn_bg_iou_thresh,
        box_score_thresh=cfg.box_score_thresh,
        box_nms_thresh=cfg.box_nms_thresh,
        box_detections_per_img=cfg.box_detections_per_img,
    )
    return model


def main():
    cfg = Config()
    if cfg.seed is not None:
        random.seed(cfg.seed)
        torch.manual_seed(cfg.seed)
        torch.cuda.manual_seed_all(cfg.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(cfg.output_dir, exist_ok=True)

    dataset = CocoDetectionMapped(
        cfg.images_dir, cfg.annotations_json, transforms=build_transforms(cfg)
    )
    data_loader = DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=collate_fn,
    )

    model = build_model(cfg, num_classes=dataset.num_classes)
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(
        params, lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay
    )

    # Warmup + cosine decay to min_lr
    def lr_lambda(epoch):
        if epoch < cfg.warmup_epochs:
            return float(epoch + 1) / float(cfg.warmup_epochs)
        progress = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
        cosine = 0.5 * (1.0 + torch.cos(torch.tensor(progress * 3.14159265))).item()
        return max(cfg.min_lr / cfg.lr, cosine)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    scaler = torch.cuda.amp.GradScaler() if cfg.amp and device.type == "cuda" else None

    for epoch in range(cfg.epochs):
        avg_loss = train_one_epoch(model, optimizer, data_loader, device, scaler, cfg.accumulate)
        scheduler.step()

        ckpt_path = os.path.join(cfg.output_dir, f"fasterrcnn_manual_epoch_{epoch + 1}.pth")
        torch.save(model.state_dict(), ckpt_path)

        # simple log
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"Epoch [{epoch+1}/{cfg.epochs}]  loss={avg_loss:.4f}  lr={lr_now:.6g}  saved={ckpt_path}")

    print("Training complete.")


if __name__ == "__main__":
    main()

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


C:\Users\asus-\AppData\Local\Temp\ipykernel_18144\2601951631.py:432: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if cfg.amp and device.type == "cuda" else None
C:\Users\asus-\AppData\Local\Temp\ipykernel_18144\2601951631.py:328: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch [1/200]  loss=1.8437  lr=0.00133333  saved=E:\code\training code\runs-fasterrcnn-manual\fasterrcnn_manual_epoch_1.pth
Epoch [2/200]  loss=1.3325  lr=0.002  saved=E:\code\training code\runs-fasterrcnn-manual\fasterrcnn_manual_epoch_2.pth
Epoch [3/200]  loss=0.9796  lr=0.002  saved=E:\code\training code\runs-fasterrcnn-manual\fasterrcnn_manual_epoch_3.pth
Epoch [4/200]  loss=0.6958  lr=0.00199987  saved=E:\code\training code\runs-fasterrcnn-manual\fasterrcnn_manual_epoch_4.pth
Epoch [5/200]  loss=0.6393  lr=0.00199949  saved=E:\code\training code\runs-fasterrcnn-manual\fasterrcnn_manual_epoch_5.pth
Epoch [6/200]  loss=0.7962  lr=0.00199886  saved=E:\code\training code\runs-fasterrcnn-manual\fasterrcnn_manual_epoch_6.pth
Epoch [7/200]  loss=0.7316  lr=0.00199797  saved=E:\code\training code\runs-fasterrcnn-manual\fasterrcnn_manual_epoch_7.pth
Epoch [8/200]  loss=0.6748  lr=0.00199682  saved=E:\code\training code\runs-fasterrcnn-manual\fasterrcnn_manual_epoch_8.pth
Epoch [9/200]  los